## Preparação do Ambiente

In [1]:
# Manipulação de dados
import re
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt

# NLP - NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer

# Recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("rslp", quiet=True)

# Preparação e validação dos dados
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    learning_curve
)

# Representação textual
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelos de Machine Learning
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# Pipeline
from sklearn.pipeline import Pipeline

# Avaliação
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


## Leitura da Base de Dados

In [2]:
dados = pd.read_csv("../data/dados_spam.csv", index_col=0)
dados.head()

,categoria,texto
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# 1. Pré-Processamento dos Dados

### 1.1 Limpeza e Normalização do Texto

In [5]:
ABBREVIATIONS = {
    # Conversa / SMS
    "pls": "please", "plz": "please", "thx": "thanks", "ty": "thank you", "u": "you", "ur": "your", "r": "are",
    "ya": "you", "msg": "message", "txt": "text", "idk": "i do not know", "imo": "in my opinion", "imho": "in my humble opinion",
    "asap": "as soon as possible", "rn": "right now", "btw": "by the way", "fyi": "for your information", "brb": "be right back",
    "ttyl": "talk to you later", "lol": "laughing out loud", "omg": "oh my god",
    # Tempo / urgência
    "tmr": "tomorrow", "tmrw": "tomorrow", "tdy": "today", "2day": "today", "2nite": "tonight", "nite": "night", "wk": "week",
    "wks": "weeks", "hr": "hour", "hrs": "hours", "min": "minute", "mins": "minutes",
    # Financeiro
    "acct": "account", "accts": "accounts", "txn": "transaction", "txns": "transactions", "transac": "transaction", "amt": "amount",
    "bal": "balance", "avl": "available", "stmt": "statement", "stmts": "statements", "ref": "reference",
    # Pagamentos
    "pymt": "payment", "pymts": "payments", "pmt": "payment", "pmts": "payments", "paymt": "payment",
    # Promoções / spam
    "promo": "promotion", "promos": "promotions", "disc": "discount", "pct": "percent", "exp": "expires", "expd": "expired",
    "offer": "offer", "offers": "offers",
    # Contato
    "ph": "phone", "tel": "telephone", "mob": "mobile", "addr": "address", "info": "information",
}

In [6]:
def limpar_texto(texto):

    # 1. Lowercase
    texto = texto.lower()
    # 2. Remover HTML
    texto = re.sub(r'<[^>]+>', " ", texto)
    # 3. Remover URLs
    texto = re.sub(r'https?://\S+|www\.\S+', " ", texto)
    # 4. Remover E-mails
    texto = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', " ", texto)
    # 5. Remover telefones
    texto = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', " ", texto)
    # 6. Remover Dinheiro
    texto = re.sub(r'(r\$|\$|€|£)\s?\d+(?:[.,]\d+)?', " ", texto)
    # 7. Substituindo as abreviações
    palavras = [ABBREVIATIONS.get(p,p) for p in texto.split()]
    texto = " ".join(palavras)
    # 7. Remove caracteres indesejados
    texto = re.sub(r'[^a-záàâãéêèíìîóôõöúùûüç\s.,!?]', " ", texto)
    # 8. Normaliza espaços
    texto = re.sub(r'\s+', " ", texto)

    return texto


In [7]:
df_teste = dados.copy()

In [8]:
# Realizando o tratamento como exemplo
df_teste["texto_limpo"] = df_teste["texto"].apply(limpar_texto)

df_teste[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif you oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...
3,U dun say so early hor... U c already then say...,you dun say so early hor... you c already then...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don t think he goes to usf, he lives aro..."


### 1.2 Tokenização

In [9]:
def tokenizacao(texto):

    return word_tokenize(texto, language="english")

In [10]:
df_teste["texto_token"] = df_teste["texto_limpo"].apply(tokenizacao)

df_teste[["texto_limpo", "texto_token"]].head()

,texto_limpo,texto_token
0,"go until jurong point, crazy.. available only ...","[go, until, jurong, point, ,, crazy, .., avail..."
1,ok lar... joking wif you oni...,"[ok, lar, ..., joking, wif, you, oni, ...]"
2,free entry in a wkly comp to win fa cup final ...,"[free, entry, in, a, wkly, comp, to, win, fa, ..."
3,you dun say so early hor... you c already then...,"[you, dun, say, so, early, hor, ..., you, c, a..."
4,"nah i don t think he goes to usf, he lives aro...","[nah, i, don, t, think, he, goes, to, usf, ,, ..."


In [12]:
print(df_teste["texto_token"][1])

['ok', 'lar', '...', 'joking', 'wif', 'you', 'oni', '...']


### 1.3 Remoção das StopWords (tirando partes de negação)

In [13]:
STOP_WORDS = set(stopwords.words("english"))

NEGATIONS = {
    "no", "not", "nor", "never", "neither",
    "none", "nobody", "nothing", "nowhere",
    "cannot", "without",

    # Contrações negativas
    "don't", "doesn't", "didn't",
    "isn't", "aren't", "wasn't", "weren't",
    "haven't", "hasn't", "hadn't",
    "won't", "wouldn't", "can't", "couldn't",
    "shouldn't", "mustn't", "needn't",
}

STOP_WORDS_EN = STOP_WORDS - NEGATIONS

In [14]:
def stop_words(tokens):
    resultado = []
    for token in tokens:
        if (token.isalpha() and token not in STOP_WORDS_EN):
            resultado.append(token)
    
    return resultado

In [15]:
df_teste["texto_stop_word"] = df_teste["texto_token"].apply(stop_words)

df_teste[["texto_token", "texto_stop_word"]].head()

,texto_token,texto_stop_word
0,"[go, until, jurong, point, ,, crazy, .., avail...","[go, jurong, point, crazy, available, bugis, n..."
1,"[ok, lar, ..., joking, wif, you, oni, ...]","[ok, lar, joking, wif, oni]"
2,"[free, entry, in, a, wkly, comp, to, win, fa, ...","[free, entry, wkly, comp, win, fa, cup, final,..."
3,"[you, dun, say, so, early, hor, ..., you, c, a...","[dun, say, early, hor, c, already, say]"
4,"[nah, i, don, t, think, he, goes, to, usf, ,, ...","[nah, think, goes, usf, lives, around, though]"


### 1.4 Stemmer

In [16]:
stemmer = SnowballStemmer("english")

def aplicar_stemmer(tokens):
    resultado = []
    for token in tokens:
        resultado.append(stemmer.stem(token))
    
    return resultado

In [17]:
df_teste["texto_stemmer"] = df_teste["texto_stop_word"].apply(aplicar_stemmer)

df_teste[["texto_stop_word", "texto_stemmer"]].head()

,texto_stop_word,texto_stemmer
0,"[go, jurong, point, crazy, available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,"[ok, lar, joking, wif, oni]","[ok, lar, joke, wif, oni]"
2,"[free, entry, wkly, comp, win, fa, cup, final,...","[free, entri, wkli, comp, win, fa, cup, final,..."
3,"[dun, say, early, hor, c, already, say]","[dun, say, earli, hor, c, alreadi, say]"
4,"[nah, think, goes, usf, lives, around, though]","[nah, think, goe, usf, live, around, though]"


### 1.5 Pipeline Completo

In [18]:
def pipeline_completo(texto):

    texto = limpar_texto(texto)
    token = tokenizacao(texto)
    stop_word = stop_words(token)
    stemmer = aplicar_stemmer(stop_word)

    return " ".join(stemmer)

In [19]:
dados["texto_tratado"] = dados["texto"].apply(pipeline_completo)

dados.head()

,categoria,texto,texto_tratado
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


# 2. Preparação dos Dados

### 2.1 Divisão dos Dados

In [20]:
X = dados["texto_tratado"]
y = dados["categoria"]

In [21]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

print(f"Total Treino -> {len(X_treino)}")
print(f"Total Teste -> {len(X_teste)}")

Total Treino -> 4459
Total Teste -> 1115


### 2.2 Vetorização com TF-IDF

In [22]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    min_df=5,
    max_df=0.85,
    max_features=10000,
    sublinear_tf=True
)

In [23]:
X_tfidf = vectorizer.fit_transform(X_treino)
print(X_tfidf.shape)

(4459, 1757)


In [24]:
features_names = vectorizer.get_feature_names_out()

print(f"Quantidade de Features: {len(features_names)}")
print(features_names[:50])

Quantidade de Features: 1757
['abiola' 'abl' 'abt' 'ac' 'access' 'account' 'account statement' 'across'
 'across sea' 'act' 'activ' 'actual' 'ad' 'add' 'address' 'admir'
 'admir look' 'adult' 'advanc' 'advic' 'affair' 'affect' 'aft' 'afternoon'
 'aftr' 'age' 'ago' 'ah' 'aha' 'ahead' 'aight' 'air' 'aiyah' 'aiyo' 'al'
 'alert' 'alex' 'almost' 'alon' 'alreadi' 'alright' 'alrit' 'also' 'alway'
 'amaz' 'amount' 'amp' 'an' 'angri' 'announc']


In [25]:
doc = X_tfidf[0].toarray().flatten()

top_idx = np.argsort(doc)[-10:][::-1]

for i in top_idx:
    if doc[i] > 0:
        print(f"{features_names[i]:20} {doc[i]:.3f}")

hi                   0.526
hi hi                0.488
way home             0.478
way                  0.318
home                 0.289
ok                   0.267


# 3. Modelos de NLP

### 3.1 Modelo Naive Bayes

In [30]:
pipeline_nb = Pipeline([
    ("vectorizer", vectorizer),
    ("modelo_nb", MultinomialNB(alpha=0.5))
])

pipeline_nb.fit(X_treino, y_treino)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('modelo_nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U4](2,)","['ham','spam']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.85
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",5
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True


In [31]:
previsao_nb = pipeline_nb.predict(X_teste)

print("=== RESULTADOS NAIVE BAYES ===")
print(classification_report(y_teste, previsao_nb))

=== RESULTADOS NAIVE BAYES ===
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.97      0.85      0.91       149

    accuracy                           0.98      1115
   macro avg       0.97      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

